# EEG-ERP Preprocessing 
## Batch script

This script will run on all subjects in `rawdata`. Steps performed include:
- filtering
- epoching
- mark (but don't correct) bad segments of data (trials/channels) using AutoReject
- pass the marked data to ICA
- auto-idenitfy and remove ocular indepdent components 
- apply ICA to epoched data
- apply AutoReject to ICA-cleaned data
- rereference
- export clean epochs to `.fif` file in `derivatives/erp_preprocessing`
- save an MNE report with details/vizualizations of the above steps in `derivatives/erp_preprocessing/logs`

Most parameters that you would want to change are read from `config.json`. 

It is recommended that you not apply any baseline correction at this stage (and the defult config.json file reflects this). Baseline correction can be applied in subsequent scripts, but doing baseline correction here precludes later using baseline regression.

Bad channels and non-ocular independent components can be manually identified after this script is run the first time. These can be stored in additional config files saved in `rawdata`, and then this script can be re-run to have those excluded. In an ideal world with reasnably clean data this should not be necessary, but e.g., if there are broken electrodes, EGI data, data from children, etc. it may be necessary.

---
Copyright 2026 [Aaron J Newman](https://github.com/aaronjnewman), [NeuroCognitive Imaging Lab](http://ncil.science), [Dalhousie University](https://dal.ca)

Released under the [The 3-Clause BSD License](https://opensource.org/licenses/BSD-3-Clause)

---

In [1]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import os.path as op
from os import remove
from glob import glob
from pathlib import Path
import json
import yaml
from yaml import Loader
import numpy as np
import warnings
import mne
from mne.preprocessing import annotate_amplitude
mne.set_log_level('error')
from mne_bids import BIDSPath, read_raw_bids
from scipy.stats import zscore
from autoreject import Ransac, get_rejection_threshold, AutoReject
from time import time

# Suppress nperseg warnings from PSD calculations
warnings.filterwarnings('ignore', message='nperseg.*is greater than input length', category=UserWarning)

## Read Parameters from config



In [2]:
# this shouldn't change if you run this script from its default location in code/import
bids_root = '../..'

config_file = op.join(bids_root, 'config.json')
config = json.load(open(config_file))

study_name = config['Study']['Name']
task_name = config['Study']['TaskName']
data_type = config['EEG']['data_type']
raw_extn = config['EEG']['raw_extn']
eog = config['EEG']['eog']

if config['Preprocessing']['drop_ch'] != None:
    drop_ch = config['Preprocessing']['drop_ch']

montage_fname = config['EEG']['montage']

# fix per changes to config
n_jobs = config['Preprocessing']['n_jobs']
filt_p = config['Preprocessing']['Filter']
ica_p = config['Preprocessing']['ICA']
epoch_p = config['Preprocessing']['Epoch']
epochs_suffix = config['EEG']['epochs_suffix']
conditions = config['Analysis']['conditions']

### Clean up imported parameters

Python `None` values are imported as `null` in JSON, but we want them to be `None` in Python. This function recursively replaces all `null` values with `None`.

In [3]:
def normalize_config_value(value):
    """Convert stringified config values (e.g., 'None', '[...]', '{...}') to Python objects."""
    if not isinstance(value, str):
        return value

    value_str = value.strip()
    if value_str.lower() in ('none', 'null', ''):
        return None

    import ast
    try:
        return ast.literal_eval(value_str)
    except (ValueError, SyntaxError):
        return value

### Paths

In [4]:
raw_path = op.join(bids_root)

derivatives_path = op.join(bids_root, 'derivatives', '1_erp_preprocessing')
if Path(derivatives_path).exists() == False:
    Path(derivatives_path).mkdir(parents=True)

report_path = op.join(derivatives_path, 'logs')
if Path(report_path).exists() == False:
    Path(report_path).mkdir(parents=True)

### Subject list

In [5]:
prefix = 'sub-'
subjects = sorted([s[-7:] for s in glob(raw_path + '/' + prefix + '*')])
subjects

['sub-001',
 'sub-002',
 'sub-003',
 'sub-004',
 'sub-005',
 'sub-006',
 'sub-007',
 'sub-008',
 'sub-009',
 'sub-010',
 'sub-011',
 'sub-012']

## Read manually-marked independent components
Run this script once, then inspect ICs (in the `sub-xxx.html` file that is stored in `derivatives/erp_preprocessing/reports` folder). Based on this, make decisions about whether any additional ICs should be added, or any automatically-removed ICs should be included. Add these to the `participants_manual_ic.yml` file located in the present folder. Then, run this script again to apply the changes.

Additional ICs to remove were selected based on:
- participants for whom more than 15% of trials were removred by AutoReject after ICA correct
- participants for whom the average across all trials and all electrodes did not show a clear pattern of P1-N1-P2 components
- for such participants, the scalp map and details of each IC were visuall inspected. Components were removed if they were
    - focal at a single electrode, or a very low number of electrodes
    - focal at the edges of the electrode montage
    - present on a low number of trials
    - showed no systematic pattern across trials that was time-locked to stimulus onset
    
ICs to add (un-remove) were selected based on:
- IC shows clear and consistent temporal dependency on stimulus onset
- IC appears to contain P1-N1-P2 complex, or part thereof
- IC has broad scalp distribution across many electrodes, characteristic of ERP component

In [6]:
ica_manual = {'sub-001':
                        {'ses-001':
                                {'manually_excluded_ic': [0],
                                 'manually_included_ic': [2]
                                 }
                        },
               'sub-002':
                        {'ses-001':
                                {'manually_included_ic': [0]}
                        },                        
               'sub-005':
                        {'ses-001':
                                {'manually_included_ic': [8]}
                        }, 
                'sub-006':
                        {'ses-001':
                                {'manually_included_ic': [7, 8]}
                        }, 
               'sub-007':
                        {'ses-001':
                                {'manually_included_ic': [8]}
                        }, 
               'sub-008':
                        {'ses-001':
                                {'manually_included_ic': [9]}
                        }, 
               'sub-009':
                        {'ses-001':
                                {'manually_excluded_ic': [0]}
                        },                 
               'sub-010':
                        {'ses-001':
                                {'manually_included_ic': [5],
                                 'manually_excluded_ic': [8]
                                 }
                        },                         
               'sub-012':
                        {'ses-001':
                                {'manually_excluded_ic': [6]}
                        }, 
                }

## Manually specify bad channels for rejection
Run this script once, then inspect the `sub-xxx.html` file that is stored in `derivatives/erp_preprocessing/reports` folder. Based on this, make decisions about whether any additional channels should be marked
as bad prior to AutoReject and ICA


In [7]:
bad_channels_manual = {'sub-005': ['Fz'], 
                       'sub-007': ['Fz']
                       }

## Handle study Metadata

Read behavioural log file to obtain trial metadata. This will need editing (or simply ignore) for every study

In [8]:
use_metadata = False  # change to True if you have behavioural data you want to add to the EEG data output by this script

# list columns in log file that we do not want to include in the metadata
# add to this according to your study; typically there are numerous columns that are not relevant to the EEG data
beh_drop_cols = ['frameRate', 'expName',
                 'session', 'participant'
                 ]

## Preprocessing Functions
### Parse BIDS parts

In [9]:
def parse_bids_parts(f, default_task, default_run='01'):
    """Extract task and run labels from a BIDS filename."""
    bids_parts = f.split('/')[-1].split('_')
    try:
        task = [t for t in bids_parts if 'task' in t][0].split('-')[-1]
    except:
        task = default_task
    try:
        run = [r for r in bids_parts if 'run' in r][0].split('-')[-1]
    except:
        run = default_run
    return task, run

### Load raw

In [10]:
def load_raw(raw_path, subject, ses, task, run, data_type):
    """Initialize BIDSPath and load one raw recording."""
    in_path = BIDSPath(root=raw_path,
                       subject=subject[-3:],
                       session=ses,
                       task=task,
                       run=run,
                       datatype=data_type)
    raw = read_raw_bids(in_path)
    return raw

### Process events

In [11]:
def process_events(raw, report):
    """Parse events, normalize annotation labels, and add mapped events to report."""
    # Read trigger mapping table from this script folder (with fallback to BIDS-root-relative path).
    trigger_map_file = op.join('trigger_mapping_v2.txt')
    if not op.exists(trigger_map_file):
        trigger_map_file = op.join(bids_root, 'code', '2_preprocessing_erp', 'trigger_mapping_v2.txt')

    trigger_map = pd.read_csv(trigger_map_file, sep='|', engine='python')
    trigger_map.columns = [c.strip() for c in trigger_map.columns]
    trigger_map = trigger_map[['Code', 'Category', 'POS']].copy()
    trigger_map['Code'] = pd.to_numeric(trigger_map['Code'], errors='coerce')
    trigger_map = trigger_map.dropna(subset=['Code'])
    trigger_map['Code'] = trigger_map['Code'].astype(int)
    trigger_map['Category'] = trigger_map['Category'].astype(str).str.strip()
    trigger_map['POS'] = trigger_map['POS'].astype(str).str.strip()

    # Build mapping from canonical stimulus labels (e.g., 'S  1') to '<Category>/<POS>'.
    slabel_to_category_pos = {
        f"S{code:>3}": f"{cat}/{pos}"
        for code, cat, pos in zip(trigger_map['Code'], trigger_map['Category'], trigger_map['POS'])
    }

    def _normalize_annotation_label(label):
        """Normalize labels such as np.str_('Stimulus/S  1') -> 'S  1'."""
        text = str(label).strip()
        if text.startswith('np.str_(') and text.endswith(')'):
            text = text[len('np.str_('):-1].strip("'\"")
        if '/' in text:
            text = text.split('/')[-1]
        return text.strip()

    # Let MNE build integer event codes from annotations, then remap labels below.
    events, event_dict = mne.events_from_annotations(raw)

    # remove event code for '__' (start-of-recording marker), including wrapped/prefixed variants
    start_marker_codes = [
        code for label, code in event_dict.items()
        if _normalize_annotation_label(label) == '__'
    ]
    if len(start_marker_codes) > 0:
        events = np.delete(events, list(np.where(np.isin(events[:, 2], start_marker_codes))[0]), axis=0)

    # Build final event_id as integer -> '<Category>/<POS>' label.
    event_id = {}
    for raw_label, event_code in event_dict.items():
        slabel = _normalize_annotation_label(raw_label)
        if slabel in slabel_to_category_pos:
            event_id[event_code] = slabel_to_category_pos[slabel]

    # remove event code == 0 and duplicate events
    events = events[np.nonzero(events[:, 2])]
    # remove rows from events where column 2 == 10001
    events = events[events[:, 2] != 10001]
    events = np.unique(events, axis=0)

    # Report needs label -> code, and only for codes still present in events.
    present_codes = set(events[:, 2].astype(int))
    event_id_report = {
        label: code for code, label in event_id.items()
        if code in present_codes
    }
    report.add_events(events, event_id=event_id_report, sfreq=raw.info['sfreq'], title='Events')
    plt.close()

    return events, event_id, report

### Annotate bad segments of data (flats)

In [12]:
def annotate_flat(raw):
    """Mark flat channels and bad segments using amplitude annotation."""
    annotations, bads = annotate_amplitude(raw, flat=0., bad_percent=25)
    raw.set_annotations(annotations)
    return raw

### Filtering

In [13]:
def filter_raw(raw, filt_p, n_jobs, report):
    """Filter raw data for ICA and for final analysis. Returns analysis-filtered raw and ICA-filtered copy."""
    picks = mne.pick_types(raw.info, eeg=True, eog=True)

    raw_ica = raw.load_data().copy().filter(filt_p['l_freq_ica'], filt_p['h_freq'],
                                            picks=picks, n_jobs=n_jobs)
    raw.filter(filt_p['l_freq'], filt_p['h_freq'], picks=picks, n_jobs=n_jobs)

    report.add_raw(raw=raw, psd=True, butterfly=True,
                   title='Raw data, bandpass filtered ' + str(filt_p['l_freq']) + '–' + str(filt_p['h_freq']))
    plt.close()

    return raw, raw_ica, report

### Clean data pre-ICA with AutoReject

In [14]:
def autoreject_pre_ica(raw_ica, events, event_id, epoch_p, ica_p, n_jobs, report):
    """Epoch ICA-filtered data and fit AutoReject to identify bad epochs and channels."""
    # Convert external event_id (int -> label) to MNE format (label -> int).
    event_id_mne = {label: code for code, label in event_id.items()}

    # Normalize baseline if config stored it as a string.
    baseline = epoch_p['baseline']
    if isinstance(baseline, str):
        baseline_str = baseline.strip()
        if baseline_str.lower() in ('none', 'null', ''):
            baseline = None
        else:
            import ast
            baseline = ast.literal_eval(baseline_str)
    if baseline is not None:
        baseline = tuple(baseline)

    epochs_ica = mne.Epochs(raw_ica, events, event_id_mne,
                            epoch_p['tmin'], epoch_p['tmax'],
                            baseline=baseline, detrend=epoch_p['detrend'],
                            reject=None, 
                            preload=True)

    ar = AutoReject(n_interpolate=[1, 2, 4, 8],
                    random_state=ica_p['ica_random_state'],
                    picks=mne.pick_types(epochs_ica.info, eeg=True, eog=False),
                    n_jobs=n_jobs, verbose=False)
    ar.fit(epochs_ica)
    plt.close()

    print('n_interpolate = ' + str(ar.n_interpolate_['eeg']))

    reject_log = ar.get_reject_log(epochs_ica)
    fig = reject_log.plot('horizontal', show=False)
    report.add_figure(fig=fig, title='AutoReject log (pre-ICA)')
    plt.close()

    return epochs_ica, ar, reject_log, report

### Fit ICA and identify ocular components

In [15]:
def fit_ica(raw_ica, epochs_ica, reject_log, ica_p, epoch_p, subject, ses, ica_manual, n_jobs, report):
    """Fit ICA, identify EOG components with adaptive z-threshold, apply manual IC overrides."""

    def _n_components_candidates(n_components):
        """Return retry candidates for explained-variance mode n_components in (0, 1)."""
        if isinstance(n_components, float) and 0 < n_components < 1:
            candidates = [n_components]
            # Escalate toward 1.0: 0.99, 0.999, 0.9999, ...
            for nines in range(2, 9):
                cand = 1 - 10 ** (-nines)
                if cand > candidates[-1]:
                    candidates.append(cand)
            return candidates
        return [n_components]

    fit_epochs = epochs_ica[~reject_log.bad_epochs]
    n_components_candidates = _n_components_candidates(ica_p['n_components'])
    last_ica_error = None

    for n_components_try in n_components_candidates:
        ica = mne.preprocessing.ICA(method=ica_p['ica_method'],
                                    n_components=n_components_try,
                                    random_state=ica_p['ica_random_state'],
                                    max_iter='auto')
        try:
            ica.fit(fit_epochs, decim=3, picks=['eeg'])
            break
        except RuntimeError as err:
            err_msg = str(err)
            if 'threshold results in 1 component' in err_msg and n_components_try != n_components_candidates[-1]:
                print(f"ICA fit failed with n_components={n_components_try}. Retrying with higher explained variance threshold...")
                last_ica_error = err
                continue
            raise
    else:
        raise RuntimeError(
            f"ICA fit failed for all n_components retries: {n_components_candidates}. "
            f"Last error: {last_ica_error}"
        )

    # adaptive z-threshold: step down until at least n_max_eog components are found
    ica.exclude = []
    num_excl = 0
    z_thresh = ica_p['ica_zthresh']
    z_step = ica_p['ica_zstep']

    while num_excl < ica_p['n_max_eog'] and z_thresh > 0:
        eog_indices, eog_scores = ica.find_bads_eog(epochs_ica, threshold=z_thresh)
        num_excl = len(eog_indices)
        z_thresh -= z_step

    # Convert eog_indices from np.int64 to regular Python ints
    ica.exclude = [int(x) for x in eog_indices]

    # apply manual IC overrides if defined for this subject/session
    # Convert ses (e.g., '001') to BIDS session format (e.g., 'ses-001')
    ses_key = 'ses-' + ses
    if subject in ica_manual and ses_key in ica_manual[subject]:
        if 'manually_excluded_ic' in ica_manual[subject][ses_key]:
            for ic in ica_manual[subject][ses_key]['manually_excluded_ic']:
                if ic not in ica.exclude:
                    ica.exclude.append(ic)
                    print(f"Manually excluding IC {ic} for {subject} {ses_key}")
                else:
                    print(f"IC {ic} already in exclude list for {subject} {ses_key}")
        if 'manually_included_ic' in ica_manual[subject][ses_key]:
            for ic in ica_manual[subject][ses_key]['manually_included_ic']:
                if ic in ica.exclude:
                    ica.exclude.remove(ic)
                    print(f"Manually including IC {ic} for {subject} {ses_key}")
                else:
                    print(f"IC {ic} not in exclude list (already included) for {subject} {ses_key}")

    print(f"Excluding ICs {ica.exclude} (z-threshold={z_thresh + z_step:.2f})")
    
    report.add_ica(ica=ica, title='ICA', inst=fit_epochs,
                   eog_scores=eog_scores, n_jobs=n_jobs)
    plt.close()

    return ica, eog_scores, report

### Create final epochs, apply ICA then AutoReject

In [16]:
def epoch_apply_ica_autoreject(raw, events, event_id, epoch_p, ica, ar, report):
    """Epoch analysis-filtered data, apply ICA correction, and run post-ICA AutoReject."""
    # Convert external event_id (int -> label) to MNE format (label -> int).
    event_id_mne = {label: code for code, label in event_id.items()}

    baseline = normalize_config_value(epoch_p['baseline'])
    if baseline is not None:
        baseline = tuple(baseline)

    reject = normalize_config_value(epoch_p['reject'])
    flat = normalize_config_value(epoch_p['flat'])

    epochs = mne.Epochs(raw, events, event_id_mne,
                        epoch_p['tmin'], epoch_p['tmax'],
                        baseline=baseline, detrend=epoch_p['detrend'],
                        reject=reject, flat=flat,
                        preload=True)

    ica.apply(epochs)

    epochs_clean, reject_log = ar.fit_transform(epochs, return_log=True)

    fig = reject_log.plot('horizontal', show=False)
    report.add_figure(fig=fig, title='AutoReject log (post-ICA)')
    plt.close()

    return epochs, epochs_clean, reject_log, report

### Rereference and save clean epochs

In [17]:
def rereference_and_save(epochs_clean, epoch_p, derivatives_path, subject, ses, task_name, data_type, raw_extn, run, epochs_suffix):
    """Rereference epochs to average and save .fif file with JSON sidecar."""
    epochs_clean.set_eeg_reference(ref_channels=epoch_p['rereference'])

    out_path = op.join(derivatives_path, subject, data_type)
    out_file = op.join(out_path, subject + '_task-' + task_name + '_desc-preproc' + epochs_suffix)

    if not op.exists(str(out_path)):
        Path(str(out_path)).mkdir(parents=True)
    else:
        try:
            for f in glob(out_path + '/*'):
                remove(f)
        except:
            pass

    epochs_clean.save(out_file, overwrite=True)

    out_meta = {
        'Description': 'ERP epochs preprocessed by code/2_preprocessing_erp/1_preprocessing_batch.ipynb, including filtering, ICA artifact removal, AutoReject correction, and rereferencing',
        'Sources': f'bids:raw:{subject}/ses-{ses}/{data_type}/{subject}_ses-{ses}_task-{task_name}_run-0{run}_{data_type}{raw_extn}'
    }
    with open(out_file[:-4] + '.json', 'w') as outfile:
        json.dump(out_meta, outfile, indent=2)

    return out_file

### Finalize and save report

In [18]:
def finalize_report(epochs_clean, epochs, conditions, event_id, ica, ar, report, report_path, subject, proc_time):
    """Add ERP plots to report, save HTML, and return a single-row rejection stats DataFrame."""
    report.add_epochs(epochs_clean, title='Epochs')

    if len(epochs_clean) > 0:
        fig = epochs_clean.copy().average().plot(spatial_colors=True, show=False)
        report.add_figure(fig=fig, title='Grand average over all epochs')
        plt.close(fig)

        for condition in conditions:
            if condition not in epochs_clean.event_id:
                continue
            condition_epochs = epochs_clean[condition]
            if len(condition_epochs) == 0:
                continue
            fig = condition_epochs.copy().average().plot(spatial_colors=True, show=False)
            report.add_figure(fig=fig, title=str(condition))
            plt.close(fig)

    report.save(op.join(report_path, subject + '.html'), overwrite=True)

    n_epochs_total = epochs.selection.shape[0]
    rm_epochs = n_epochs_total - epochs_clean.selection.shape[0]
    pct_epochs = (rm_epochs / n_epochs_total * 100) if n_epochs_total > 0 else 0.0

    return pd.DataFrame({'id': subject,
                         'cpu_time': proc_time,
                         'ntrials_rej': rm_epochs,
                         '%t_rej': round(pct_epochs, 2),
                         'ic_rm': len(ica.exclude),
                         'n_interp': str(ar.n_interpolate_['eeg'])
                         }, index=[0])

## Main Loop

In [19]:
rej_log_list = []

for subject in subjects:
    sessions = [f.split('/')[-1][-3:] for f in glob(op.join(raw_path, subject, 'ses-*'))]
    for ses in sessions:
        in_files = sorted(glob(op.join(raw_path, subject, 'ses-' + ses, data_type, subject + '*' + raw_extn)))

        if len(in_files) == 0:
            continue

        start_time = time()
        print('\n-------------------------')
        print('-------- ' + subject + ' --------')
        print('-------------------------')

        raws = []
        for f in in_files:
            task, run = parse_bids_parts(f, task_name)
            raws.append(load_raw(raw_path, subject, ses, task, run, data_type))

        raw = mne.concatenate_raws(raws) if len(raws) > 1 else raws[0]
        report = mne.Report(subject=subject,
                            title=study_name + ' preprocessing: ' + subject + ' ' + ses,
                            verbose='WARNING')

        events, event_id, report = process_events(raw, report)

        raw = annotate_flat(raw)
        if subject in bad_channels_manual:
            raw.info['bads'].extend(bad_channels_manual[subject])

        print('Filtering...')
        job_start = time()
        raw, raw_ica, report = filter_raw(raw, filt_p, n_jobs, report)
        print('Filtering took ' + str(round(time() - job_start, 1)) + ' s')

        print('AutoReject pre-ICA...')
        job_start = time()
        epochs_ica, ar, reject_log, report = autoreject_pre_ica(raw_ica, events, event_id, epoch_p, ica_p, n_jobs, report)
        print('AutoReject took ' + str(round(time() - job_start, 1)) + ' s')

        print('ICA...')
        job_start = time()
        ica, _, report = fit_ica(raw_ica, epochs_ica, reject_log, ica_p, epoch_p, subject, ses, ica_manual, n_jobs, report)
        print('ICA took ' + str(round(time() - job_start, 3)) + ' s')

        if use_metadata:
            metadata, events = get_metadata(raw_path, subject, ses, task_name, events, beh_drop_cols)

        print('AutoReject post-ICA...')
        job_start = time()
        epochs, epochs_clean, reject_log, report = epoch_apply_ica_autoreject(raw, events, event_id, epoch_p, ica, ar, report)
        print('AutoReject took ' + str(round(time() - job_start, 1)) + ' s')

        if use_metadata:
            epochs_clean.metadata = metadata.iloc[epochs_clean.selection]

        rereference_and_save(epochs_clean, epoch_p, derivatives_path, subject, ses, task_name, data_type, raw_extn, run, epochs_suffix)

        proc_time = time() - start_time
        print('Total processing time: ' + str(round(proc_time, 1)) + ' seconds')

        rej_log_list.append(
            finalize_report(epochs_clean, epochs, conditions, event_id, ica, ar, report, report_path, subject, proc_time)
        )

rej_log = pd.concat(rej_log_list)
rej_log.to_csv(report_path + '/rejection_log_all_Ss.csv')


-------------------------
-------- sub-001 --------
-------------------------
Filtering...
Filtering took 2.3 s
AutoReject pre-ICA...


KeyboardInterrupt: 